# Phase space grid example

This notebook demonstrates the **psgrid** workflow as an alternative to the Gaussian-ensemble
approach in `example_1.ipynb`.

### The idea

For wavefront-aberration studies the quantity of interest is the **transfer function**
$p(\text{port} \mid x_0, v_0)$ — the probability of each output port given the atom's
initial phase-space coordinate. Because the simulation is deterministic for a single atom,
this requires **one atom per grid node**.

Once the map is computed, any Gaussian cloud average follows analytically:
$$
P_\text{port} = \sum_{x_0, v_0} w(x_0, v_0)\, p(\text{port} \mid x_0, v_0)
                \,\Delta x_0\,\Delta v_0
$$
and storing **complex amplitudes** lets you apply arbitrary phase profiles offline
(phase shear, phase scan) without re-running ais++.

### Port probability and the coherence criterion

A **port** is a group of wavepackets that (a) are in the same internal state and
(b) are separated by less than the coherence length at detection.
For a two-wavepacket interfering port the probability is
$$
P = A_1^2 + A_2^2 + 2 A_1 A_2 \cos(\Delta\phi)
$$
where $\Delta\phi$ is the total phase difference including the kinematic separation phase
$m\,\bar{v}\cdot\Delta x/\hbar$.
This is computed in quad precision inside ais++ and stored as `phase_shifts`.
For a non-interfering port (single wavepacket, or wavepackets too far apart)
$P = A^2$ and `phase_shifts = 0`.

In the PSR example with $\kappa_\mathrm{PSR}=3140\,\mathrm{rad/m}$, the four simulated
paths end up spatially separated by more than the coherence length, so each is its own
non-interfering port. For a standard MZ without PSR, paths that recombine at the same
output port would form an interfering port.

### What this notebook covers

1. Load the PSMAP file and inspect its structure
2. Visualise probability and phase-shift maps across the phase space grid
3. Sanity-check: probabilities sum to 1 at every grid node
4. Compute Gaussian-cloud-averaged port probabilities
5. Scan the interferometer phase **offline** using the stored $\Delta\phi$
6. Apply an offline phase shear and recompute the PSR spatial distribution
7. Scan cloud position without re-running ais++

**Setup**
```bash
pip install numpy scipy matplotlib h5py jupyter

cd examples/
ais++ -i input-files/PSR_EXAMPLE_PSGRID.aisi -o output-files/PSR_EXAMPLE_PSGRID.h5
```

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt

try:
    plt.style.use('seaborn-v0_8-ticks')
except OSError:
    plt.style.use('seaborn-ticks')

plt.rcParams.update({
    'font.family'   : 'serif',
    'font.size'     : 9,
    'axes.labelsize': 9,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'legend.frameon': False,
    'savefig.dpi'   : 150,
})

kB     = 1.380649e-23
m_Sr87 = 86.909 * 1.66054e-27
hbar   = 1.054571817e-34

## 1. Load the phase space map

The PSMAP file stores **one row per port per atom**.
Each row contains the atom's initial conditions, the port's mean final position/velocity,
its probability (correctly computed in C++ with quad precision), and the individual
wavepacket amplitudes and positions needed for offline phase manipulation.

In [ ]:
FNAME = 'output-files/PSR_EXAMPLE_PSGRID.h5'

with h5py.File(FNAME, 'r') as f:
    # Per-port quantities
    init_pos   = f['initial_positions'][:]     # (N_ports, 3)  x0,y0,z0
    init_vel   = f['initial_velocities'][:]    # (N_ports, 3)  vx0,vy0,vz0
    final_pos  = f['final_positions'][:]       # (N_ports, 3)  mean xf,yf,zf
    final_vel  = f['final_velocities'][:]      # (N_ports, 3)  mean vxf,...
    prob       = f['probabilities'][:]         # (N_ports,)    P_port
    dphi       = f['phase_shifts'][:]          # (N_ports,)    Δφ (quad-precise)
    dphi_err   = f['phase_shift_errors'][:]    # (N_ports,)
    state      = f['states'][:]                # (N_ports,)
    interfering= f['is_interfering'][:]        # (N_ports,)  1 = interfering
    atom_idx   = f['atom_indices'][:]          # (N_ports,)
    # Per-wavepacket quantities (second slot zero/empty for non-interfering ports)
    amp0       = f['amp0'][:]                  # (N_ports,)  |c_0|
    amp1       = f['amp1'][:]                  # (N_ports,)  |c_1|  (0 if non-interfering)
    wp0_pos    = f['wp0_final_positions'][:]   # (N_ports, 3)
    wp1_pos    = f['wp1_final_positions'][:]   # (N_ports, 3)
    wp0_vel    = f['wp0_final_velocities'][:]  # (N_ports, 3)
    wp1_vel    = f['wp1_final_velocities'][:]  # (N_ports, 3)
    path0 = np.array([p.decode() if isinstance(p, bytes) else p for p in f['path0'][:]])
    path1 = np.array([p.decode() if isinstance(p, bytes) else p for p in f['path1'][:]])

n_ports = len(prob)
n_atoms = int(atom_idx.max()) + 1
unique_atoms, ports_per_atom = np.unique(atom_idx, return_counts=True)
n_ports_per_atom = ports_per_atom[0]   # same for all atoms in grid mode

print(f'Total ports   : {n_ports:,}')
print(f'Atoms         : {n_atoms:,}')
print(f'Ports/atom    : {n_ports_per_atom}')
print(f'Interfering   : {interfering.sum()} / {n_ports} ports')
print(f'Max Δφ error  : {dphi_err.max():.2e} rad')
print()
# Show port structure for the first atom
mask0 = atom_idx == 0
print('Ports for atom 0:')
for i in np.where(mask0)[0]:
    flag = 'interfering' if interfering[i] else 'solo'
    print(f'  path0={path0[i]:4s}  path1={path1[i]:4s}  state={state[i]}  P={prob[i]:.4f}  {flag}')

## 2. Build structured (nx, nvx) grids

Pivot the flat port array into a `(nx, nvx, n_ports_per_atom)` array.
The grid ordering (x outermost, vx innermost) matches the ais++ grid constructor.

In [ ]:
# Initial conditions: same across all ports of one atom; read from port index 0 of each atom
first_port = np.array([np.searchsorted(atom_idx, ai) for ai in range(n_atoms)])
X0_atoms  = init_pos[first_port, 0]   # (n_atoms,)
VX0_atoms = init_vel[first_port, 0]   # (n_atoms,)

xs  = np.unique(X0_atoms)
vxs = np.unique(VX0_atoms)
nx, nvx = len(xs), len(vxs)
print(f'Grid: {nx} × {nvx}')
print(f'  x  : {xs[0]*1e6:.1f} to {xs[-1]*1e6:.1f} µm')
print(f'  vx : {vxs[0]*1e3:.2f} to {vxs[-1]*1e3:.2f} mm/s')

# Reshape (n_atoms, n_ports_per_atom) → (nx, nvx, n_ports_per_atom)
def atom_port_grid(data_1d):
    """Pivot (n_ports,) flat array → (nx, nvx, n_ports_per_atom) grid."""
    return data_1d.reshape(n_atoms, n_ports_per_atom).reshape(nx, nvx, n_ports_per_atom)

PROB   = atom_port_grid(prob)        # (nx, nvx, nP)
DPHI   = atom_port_grid(dphi)        # (nx, nvx, nP)
STATE  = atom_port_grid(state)       # (nx, nvx, nP)
INTER  = atom_port_grid(interfering) # (nx, nvx, nP)
AMP0   = atom_port_grid(amp0)        # (nx, nvx, nP)
AMP1   = atom_port_grid(amp1)        # (nx, nvx, nP)

# Final x-positions of each wavepacket (for offline shear)
XF0   = atom_port_grid(wp0_pos[:, 0])   # (nx, nvx, nP)
XF1   = atom_port_grid(wp1_pos[:, 0])   # (nx, nvx, nP)

X0_grid  = X0_atoms.reshape(nx, nvx)
VX0_grid = VX0_atoms.reshape(nx, nvx)

# Port labels from the first atom (consistent for all atoms)
port_labels = [f'{path0[first_port[0]+p]}/{path1[first_port[0]+p]}'
               for p in range(n_ports_per_atom)]
print('Port labels:', port_labels)

## 3. Probability and phase-shift maps

Plot $P(x_0, v_{x0})$ and $\Delta\phi(x_0, v_{x0})$ for each port.
The phase-shift map encodes wavefront aberrations: for a perfect flat wavefront
$\Delta\phi$ is constant across the grid.

In [ ]:
nP     = n_ports_per_atom
extent = [vxs[0]*1e3, vxs[-1]*1e3, xs[0]*1e6, xs[-1]*1e6]  # [mm/s, mm/s, µm, µm]

fig, axes = plt.subplots(2, nP, figsize=(3*nP, 5),
                         gridspec_kw=dict(hspace=0.4, wspace=0.4))
if nP == 1:
    axes = axes[:, np.newaxis]

for pi in range(nP):
    lbl = port_labels[pi]

    im0 = axes[0, pi].imshow(PROB[:, :, pi], origin='lower', extent=extent,
                              aspect='auto', cmap='viridis', vmin=0)
    axes[0, pi].set_title(f'{lbl}\nstate {STATE[0,0,pi]}', fontsize=8)
    axes[0, pi].set_xlabel(r'$v_{x0}$ [mm/s]')
    if pi == 0:
        axes[0, pi].set_ylabel(r'$x_0$ [µm]')
    plt.colorbar(im0, ax=axes[0, pi], label=r'$P$')

    if INTER[0, 0, pi]:
        cmap, vabs = 'RdBu', np.pi
        label_dphi = r'$\Delta\phi$ [rad]'
    else:
        cmap, vabs = 'Greys', 0.01
        label_dphi = r'$\Delta\phi=0$ (solo)'

    im1 = axes[1, pi].imshow(DPHI[:, :, pi], origin='lower', extent=extent,
                              aspect='auto', cmap=cmap,
                              vmin=-vabs, vmax=vabs)
    axes[1, pi].set_xlabel(r'$v_{x0}$ [mm/s]')
    if pi == 0:
        axes[1, pi].set_ylabel(r'$x_0$ [µm]')
    plt.colorbar(im1, ax=axes[1, pi], label=label_dphi)

axes[0, 0].annotate(r'$P(x_0,v_{x0})$', xy=(-0.35, 0.5), xycoords='axes fraction',
                     fontsize=9, fontweight='bold', rotation=90, va='center')
axes[1, 0].annotate(r'$\Delta\phi(x_0,v_{x0})$', xy=(-0.35, 0.5), xycoords='axes fraction',
                     fontsize=9, fontweight='bold', rotation=90, va='center')
plt.suptitle('Phase space transfer functions — PSR MZ ($n=1$, $T=2.225$ s)', y=1.01)
plt.savefig('psmap_maps.png', bbox_inches='tight')
plt.show()

## 4. Sanity check: probabilities sum to 1

In [ ]:
total_prob = PROB.sum(axis=2)   # sum over ports at each grid node → (nx, nvx)

print('Sum of port probabilities over all ports at each grid node:')
print(f'  mean = {total_prob.mean():.6f}')
print(f'  min  = {total_prob.min():.6f}')
print(f'  max  = {total_prob.max():.6f}')
print(f'  std  = {total_prob.std():.2e}')

assert np.allclose(total_prob, 1.0, atol=1e-4), \
    'Probability normalisation violated — check amplitude threshold or path selection.'

## 5. Gaussian-cloud-averaged port probabilities

Define a Gaussian cloud $(x_\mathrm{cm}, v_{x,\mathrm{cm}}, \sigma_x, T_\mathrm{trans})$
and compute the mean probability of each port via a weighted sum over the grid.
This replaces a full $10^6$-atom Gaussian-ensemble simulation.

In [ ]:
def gaussian_weights(X0, VX0, x_cm=0., vx_cm=0., sigma_x=1e-4, T_trans=1e-9):
    sigma_vx = np.sqrt(kB * T_trans / m_Sr87)
    w = (np.exp(-0.5 * ((X0  - x_cm ) / sigma_x )**2) *
         np.exp(-0.5 * ((VX0 - vx_cm) / sigma_vx)**2))
    return w / w.sum()

def cloud_port_probs(PROB, X0, VX0, **cloud_kwargs):
    """Mean probability of each port for a Gaussian cloud. Returns (n_ports_per_atom,)."""
    w = gaussian_weights(X0, VX0, **cloud_kwargs)   # (nx, nvx)
    return np.einsum('ij,ijp->p', w, PROB)

P_ports = cloud_port_probs(PROB, X0_grid, VX0_grid)

print('Port probabilities for reference cloud (σ_x=100 µm, T=1 nK, centred):')
for lbl, p in zip(port_labels, P_ports):
    print(f'  {lbl}  →  P = {p:.4f}')
print(f'  sum = {P_ports.sum():.6f}')

## 6. Offline phase scan

Scanning a global phase offset $\delta\phi$ is equivalent to changing the final-pulse
phase of the interferometer. The modified port probability is:
$$
P(\delta\phi) = A_0^2 + A_1^2 + 2 A_0 A_1 \cos(\Delta\phi + \delta\phi)
$$
where $\Delta\phi$ is already stored with quad precision and $\delta\phi$ is a small
perturbation computed in ordinary double precision.

For **non-interfering** ports $P = A_0^2$ regardless of $\delta\phi$; only interfering
ports contribute to the fringe.

In [ ]:
def port_probs_with_phase(DPHI, AMP0, AMP1, INTER, dphi_extra):
    """
    Recompute port probabilities after adding dphi_extra to each port's phase shift.
    For non-interfering ports dphi_extra has no effect.
    Returns PROB_new with same shape as PROB: (nx, nvx, nP).
    """
    diag  = AMP0**2 + AMP1**2
    cross = 2 * AMP0 * AMP1 * np.cos(DPHI + dphi_extra)
    return diag + INTER * cross

phi_scan = np.linspace(0, 2*np.pi, 200)
w = gaussian_weights(X0_grid, VX0_grid)   # (nx, nvx)

# For each port and scan point, compute the Gaussian-weighted probability
P_scan = np.zeros((len(phi_scan), nP))
for k, dphi_extra in enumerate(phi_scan):
    PROB_k = port_probs_with_phase(DPHI, AMP0, AMP1, INTER, dphi_extra)
    P_scan[k] = np.einsum('ij,ijp->p', w, PROB_k)

fig, ax = plt.subplots(figsize=(5.5, 3))
for pi, lbl in enumerate(port_labels):
    ls = '-' if STATE[0,0,pi] == 0 else '--'
    ax.plot(phi_scan / np.pi, P_scan[:, pi], ls=ls, label=lbl)

ax.set_xlabel(r'Added phase $\delta\phi / \pi$')
ax.set_ylabel('Mean port probability')
ax.set_title('Offline phase scan — no ais++ re-run needed')
ax.set_xlim(0, 2)
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.savefig('psmap_phase_scan.png', bbox_inches='tight')
plt.show()

# Fringe contrast from peak-to-peak of the sum over state-0 ports
s0_ports = np.array([STATE[0,0,pi]==0 for pi in range(nP)])
P0_total = P_scan[:, s0_ports].sum(axis=1)
contrast = (P0_total.max() - P0_total.min()) / (P0_total.max() + P0_total.min())
print(f'State-0 fringe contrast: {contrast:.4f}')

## 7. Offline phase shear — regenerate the PSR spatial distribution

For an additional phase shear $\delta\kappa$ (on top of the $\kappa_\mathrm{PSR}=3140\,\mathrm{rad/m}$
already baked into the simulation), each wavepacket acquires an extra phase
$\delta\kappa\,x_\mathrm{final}$. The modified port probability becomes
$$
P = A_0^2 + A_1^2 + 2 A_0 A_1 \cos\!\big(\Delta\phi + \delta\kappa\,(x_{f,0}-x_{f,1})\big)
$$
and the PSR spatial histogram is the atom density weighted by $P\cdot w(x_0,v_{x0})$
binned by the mean final position $\bar{x}_f$.

In [ ]:
# Mean final x-position of each port (already loaded as final_pos)
XF_mean = atom_port_grid(final_pos[:, 0])   # (nx, nvx, nP)  mean port x_final

def psr_histogram(kappa_extra=0., x_cm=0., vx_cm=0.,
                  sigma_x=1e-4, T_trans=1e-9,
                  bins=60, x_range=(-0.008, 0.008)):
    """
    Gaussian-weighted histogram of atoms at their mean port x_final,
    with an extra offline shear kappa_extra [rad/m] applied.

    For interfering ports the extra shear modifies Δφ by kappa_extra*(x_{f,0}-x_{f,1}).
    For solo ports the probability is unchanged by the shear.
    """
    w = gaussian_weights(X0_grid, VX0_grid, x_cm, vx_cm, sigma_x, T_trans)  # (nx, nvx)

    dphi_shear = kappa_extra * (XF0 - XF1)   # (nx, nvx, nP); 0 for solo ports (XF1=0)
    PROB_sheared = port_probs_with_phase(DPHI, AMP0, AMP1, INTER, dphi_shear)

    # Weight = Gaussian(x0,vx0) * P_port
    weight = w[:, :, np.newaxis] * PROB_sheared   # (nx, nvx, nP)

    hist, edges = np.histogram(
        XF_mean.ravel(),
        bins=bins, range=x_range,
        weights=weight.ravel()
    )
    xc = 0.5 * (edges[:-1] + edges[1:])
    return xc, hist


kappa_extra_vals = [0, 1000, 3140]
labels = [
    r'$\delta\kappa = 0$ (original)',
    r'$\delta\kappa = +1000\,\mathrm{rad/m}$',
    r'$\delta\kappa = +3140\,\mathrm{rad/m}$ (doubled)',
]

fig, axes = plt.subplots(len(kappa_extra_vals), 1, figsize=(5.5, 5),
                          sharex=True, gridspec_kw=dict(hspace=0.1))

for ax, kex, lbl in zip(axes, kappa_extra_vals, labels):
    xc, hist = psr_histogram(kappa_extra=kex)
    bw = (xc[1] - xc[0]) * 1e3
    ax.bar(xc * 1e3, hist, width=bw, color='#4daf4a', alpha=0.75, linewidth=0)
    ax.set_ylabel('Weighted counts')
    ax.annotate(lbl, xy=(0.02, 0.85), xycoords='axes fraction', fontsize=8)

axes[-1].set_xlabel(r'Mean final $x$ [mm]')
plt.suptitle(r'PSR spatial distribution for different offline shear values'
             '\n' r'(simulation $\kappa_\mathrm{PSR}=3140\,\mathrm{rad/m}$)', y=1.01)
plt.savefig('psmap_shear_scan.png', bbox_inches='tight')
plt.show()

## 8. Cloud-position scan — no re-simulation

Scan the cloud centre $x_\mathrm{cm}$ and recompute $P_{s=0}$ at each position
by re-weighting the precomputed grid. The full scan takes milliseconds.

In [ ]:
x_cm_scan = np.linspace(-3e-4, 3e-4, 120)   # ±3 σ_x
s0_mask = np.array([STATE[0, 0, pi] == 0 for pi in range(nP)])

P0_xcm = np.array([
    cloud_port_probs(PROB, X0_grid, VX0_grid, x_cm=xcm)[s0_mask].sum()
    for xcm in x_cm_scan
])

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(x_cm_scan * 1e6, P0_xcm)
ax.set_xlabel(r'Cloud centre $x_\mathrm{cm}$ [µm]')
ax.set_ylabel(r'$P_{s=0}$')
ax.set_title(r'State-0 probability vs cloud position — zero ais++ calls')
plt.tight_layout()
plt.savefig('psmap_xcm_scan.png', bbox_inches='tight')
plt.show()